## DTB — Divisão Territorial Brasileira (IBGE) — Bronze
Fonte: [IBGE - DTB](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/estrutura-territorial/23701-divisao-territorial-brasileira.html) | FTP: `geoftp.ibge.gov.br/.../divisao_territorial/2025/DTB_2025.zip`

- **Licença:** Dados públicos IBGE — uso livre com atribuição
- **Atualização:** anual (2024: 5.569 municípios, 2025 inclui Boa Esperança do Norte/MT)
- **Arquivo no lake:** `/Volumes/workspace/raw/IBGE/RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods` (extraído do ZIP `DTB_2025.zip`)
- **Linhagem:** `DTB_2025.zip` → `RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods` → `workspace.bronze.dtb`
- **Grão bronze:** 1 linha por município do ODS (cópia fiel, só `normalizar_colunas`)

In [0]:
%run ./_setup_dtb

In [0]:
from data_pipeline import read_ods, normalizar_colunas, save_table, add_column_comments
from metadata.metadata import DTB_COMMENTS

In [0]:
FILE_PATH = "/Volumes/workspace/raw/IBGE/RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods"
TABLE_NAME = "workspace.bronze.dtb"
# Aba do ODS — use nome exato se houver múltiplas abas (ex.: 'RELATORIO_DTB_BRASIL_2025_MUNICIPIOS')
SHEET_NAME = 0  # 0 = primeira aba
HEADER_ROW = 6  # linha 7 do ODS (0-indexed): UF | Nome_UF | Região Geográfica ... | Nome_Município

In [0]:
df_raw = read_ods(spark, FILE_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW)
print(f"Colunas originais ODS: {df_raw.columns}")
display(df_raw.limit(5))

df = normalizar_colunas(df_raw)
print(f"Colunas normalizadas: {df.columns}")
display(df.limit(5))

# TODO: copie o print acima e cole em ETL/DTB/metadata/metadata.py se faltar alguma chave
save_table(df, TABLE_NAME)

In [0]:
existing = set(spark.table(TABLE_NAME).columns)
filtered = {k: v for k, v in DTB_COMMENTS.items() if k in existing}
missing = sorted(set(DTB_COMMENTS.keys()) - existing)
extra = sorted(existing - set(DTB_COMMENTS.keys()))
if missing:
    print(f"Aviso: chaves do catálogo ausentes no DF: {missing}")
if extra:
    print(f"Aviso: colunas no DF sem comentário no catálogo (adicionar em metadata.py): {extra}")
if filtered:
    add_column_comments(spark, TABLE_NAME, filtered)
    print(f"Comentários aplicados em {len(filtered)} colunas")

In [0]:
total_rows = df.count()
distinct_rows = df.distinct().count()
print(f"Total de registros: {total_rows:,}")
print(f"Registros únicos: {distinct_rows:,}")
print(f"Duplicatas: {total_rows - distinct_rows:,}")
display(spark.sql(f"DESCRIBE TABLE {TABLE_NAME}"))